# Score pretrained detectors under the lab protocols

Scores, on the same test crops as every other model in the lab:

- the official GenD CLIP ViT-L/14 checkpoint ([yermandy/GenD_CLIP_L_14](https://huggingface.co/yermandy/GenD_CLIP_L_14), MIT, trained on FaceForensics++ c23), on every lab protocol and on `cross_celebdf`;
- the LayerNorm-tuned CLIP detector trained in `clip_layernorm_detector.ipynb` at commit b66500a (its `subject_holdout` parameters, whose training side is the one `cross_celebdf` uses);
- the frozen CLIP linear probe, trained on the `cross_celebdf` training side.

GenD was trained on face boxes grown to 1.3x; the lab's crops hold the box at 1.5x, so its inputs are center-cropped with `tighten_crop` (both framings are reported on `cross_celebdf`).

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN` and access to both private datasets.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
CELEBDF_DATASET = "paulohenriquevn/face-decision-celebdf-test"
FFPP_DATASET = "paulohenriquevn/face-decision-ffpp"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # reports go to HF_MODEL/pretrained/<detector>/<protocol>.json
CLIP_MODEL = "openai/clip-vit-large-patch14"
GEND_MODEL = "yermandy/GenD_CLIP_L_14"
GEND_FACE_SCALE = 1.3
LAB_PROTOCOLS = ("subject_holdout", "unseen_text2img", "unseen_insight", "unseen_inpainting")

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Data

In [ ]:
from face_decision_lab.hub import load_hub_samples

samples, hub_images = load_hub_samples(HF_DATASET)
images = dict(zip([s.image_path for s in samples], hub_images, strict = True))
for prefix, repo in (("celebdf", CELEBDF_DATASET), ("faceforensics", FFPP_DATASET)):
    extra, extra_images = load_hub_samples(repo, key_prefix = prefix)
    images.update(zip([s.image_path for s in extra], extra_images, strict = True))
    samples += extra
    print(len(extra), "crops from", prefix)
print(len(samples), "crops")

### Scoring helpers

Every detector maps a list of samples to attack probabilities; `report` scores them with the shared protocol and uploads the result.

In [ ]:
import importlib.util, io, json
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from safetensors.torch import load_file
from torch import nn
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection
from face_decision_lab.faces import tighten_crop
from face_decision_lab.protocol import evaluation_report, protocol_split
from face_decision_lab.rows import open_image

api = HfApi()
pool = ThreadPoolExecutor(max_workers = 8)
KEYS = ("samples", "auc", "eer", "bpcer_at_max_apcer", "videos", "video_auc", "apcer_by_generator")

def batched_scores(part, prepare, score_batch, batch_size = 128):
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), batch_size), leave = False):
            out += score_batch(list(pool.map(prepare, part[i:i + batch_size])))
    return out

def report(detector, protocol, part, scores, **extra):
    result = {k: v for k, v in evaluation_report(part, scores).items() if k in KEYS} | extra
    api.upload_file(path_or_fileobj = json.dumps(result, indent = 2).encode(), repo_id = HF_MODEL,
                    path_in_repo = f"pretrained/{detector}/{protocol}.json", commit_message = f"{detector} on {protocol}")
    print(detector, protocol, json.dumps({k: result[k] for k in ("auc", "eer", "bpcer_at_max_apcer", "video_auc")}))
    return result

### Official GenD

`GenD.from_pretrained` fails under transformers 5, which builds models on the meta device where GenD's `__init__` cannot load CLIP, so the model is built directly and its weights loaded from `model.safetensors`. The model card's two example frames (a Deepfakes fake, then a real frame) confirm which logit means fake.

In [ ]:
spec = importlib.util.spec_from_file_location("modeling_gend", hf_hub_download(GEND_MODEL, "modeling_gend.py"))
modeling_gend = importlib.util.module_from_spec(spec)
spec.loader.exec_module(modeling_gend)
gend = modeling_gend.GenD(modeling_gend.GenDConfig.from_pretrained(GEND_MODEL))
loaded = gend.load_state_dict(load_file(hf_hub_download(GEND_MODEL, "model.safetensors")), strict = False)
assert not loaded.missing_keys and not loaded.unexpected_keys, loaded
gend = gend.cuda().eval()

examples = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
            "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
with torch.no_grad():
    x = torch.stack([gend.feature_extractor.preprocess(Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB")) for u in examples])
    fake_probability = gend(x.cuda()).softmax(-1)[:, 1].tolist()
assert fake_probability[0] > 0.5 > fake_probability[1], fake_probability

def gend_scores(part, face_scale = GEND_FACE_SCALE):
    def prepare(sample):
        return gend.feature_extractor.preprocess(tighten_crop(open_image(images[sample.image_path]), face_scale))

    return batched_scores(part, prepare, lambda b: gend(torch.stack(b).cuda()).float().softmax(-1)[:, 1].cpu().tolist())

gend_by_protocol = {}
for protocol in LAB_PROTOCOLS + ("cross_celebdf", "ffpp"):
    _, _, test = protocol_split(samples, protocol)
    gend_by_protocol[protocol] = gend_scores(test)
    report("gend_clip_l14", protocol, test, gend_by_protocol[protocol], face_scale = GEND_FACE_SCALE)
_, _, celebdf_test = protocol_split(samples, "cross_celebdf")
report("gend_clip_l14_face_scale_1.5", "cross_celebdf", celebdf_test, gend_scores(celebdf_test, 1.5), face_scale = 1.5)

### LayerNorm-tuned CLIP v1 and the frozen CLIP probe

The v1 detector (commit b66500a) used the projected 768-dimensional image embedding and a one-logit head; this class rebuilds it to load its saved parameters, which the fusion cell below scores. The frozen probe trains on the `cross_celebdf` training side.


In [ ]:
processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)
def clip_pixels(sample):
    return processor(images = open_image(images[sample.image_path]), return_tensors = "pt")["pixel_values"][0]

class LayerNormTunedCLIPv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.projection_dim, 1)

    def forward(self, pixels):
        features = nn.functional.normalize(self.encoder(pixel_values = pixels).image_embeds, dim = -1)
        return self.head(features).squeeze(-1)


In [ ]:
from face_decision_lab.baselines import LinearProbe

frozen = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL).cuda().eval()
def embed(batch):
    return list(nn.functional.normalize(frozen(pixel_values = torch.stack(batch).cuda()).image_embeds.float(), dim = -1).cpu().numpy())

train, _, _ = protocol_split(samples, "cross_celebdf")
train_features = np.stack(batched_scores(train, clip_pixels, embed, batch_size = 256))
test_features = np.stack(batched_scores(celebdf_test, clip_pixels, embed, batch_size = 256))
probe = LinearProbe().fit(train_features, np.array([not s.is_bona_fide for s in train]))
report("frozen_clip_probe", "cross_celebdf", celebdf_test, probe.attack_scores(test_features).tolist(), training_rows = len(train))

### Score fusion: v1 and GenD

The two detectors fail on each other's domain, so the fused attack score is the larger of the two probabilities: a bona fide capture has to look bona fide to both, and an attack has to fool both. Each lab protocol uses the v1 parameters trained for it; `cross_celebdf` uses the `subject_holdout` ones, whose training side it shares. No parameter is fitted to the test side.

In [ ]:
def v1_scores(checkpoint, part):
    model = LayerNormTunedCLIPv1()
    model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, checkpoint)), strict = False)
    model = model.cuda().eval()
    return batched_scores(part, clip_pixels, lambda b: torch.sigmoid(model(torch.stack(b).cuda()).float()).cpu().tolist())

for protocol in LAB_PROTOCOLS + ("cross_celebdf",):
    _, _, test = protocol_split(samples, protocol)
    checkpoint = f"{'subject_holdout' if protocol == 'cross_celebdf' else protocol}/trained_parameters.pt"
    v1 = v1_scores(checkpoint, test)
    report("layernorm_clip_v1", protocol, test, v1, checkpoint = checkpoint)
    fused = np.maximum(np.array(v1), np.array(gend_by_protocol[protocol])).tolist()
    report("fusion_max_v1_gend", protocol, test, fused, v1_checkpoint = checkpoint, gend_face_scale = GEND_FACE_SCALE)

### Learned fusion: v1 and GenD

A logistic regression over the two detectors' logits (`face_decision_lab.fusion`), fitted on calibration crops from both domains: the lab calibration side of each protocol and the FaceForensics++ validation split, weighted so every domain and class counts the same. Celeb-DF never enters the fit. The `subject_holdout` fusion is also scored on `cross_celebdf` and on the FF++ test split, since its v1 checkpoint is the one trained on their shared lab training side.

In [ ]:
from face_decision_lab.fusion import ScoreFusion, balancing_weights

_, ffpp_val, ffpp_test = protocol_split(samples, "ffpp")
gend_ffpp_val = gend_scores(ffpp_val)

for protocol in LAB_PROTOCOLS:
    _, lab_calibration, test = protocol_split(samples, protocol)
    checkpoint = f"{protocol}/trained_parameters.pt"
    calibration = lab_calibration + ffpp_val
    calibration_scores = np.column_stack([
        v1_scores(checkpoint, calibration),
        gend_scores(lab_calibration) + gend_ffpp_val,
    ])
    fusion = ScoreFusion().fit(calibration_scores, np.array([not s.is_bona_fide for s in calibration]),
                               sample_weight = balancing_weights(calibration))
    print(protocol, "logit weights (v1, GenD):", fusion.weights)
    targets = [(protocol, test)]
    if protocol == "subject_holdout":
        targets += [("cross_celebdf", protocol_split(samples, "cross_celebdf")[2]), ("ffpp", ffpp_test)]
    for name, part in targets:
        scores = np.column_stack([v1_scores(checkpoint, part), gend_by_protocol[name]])
        report("fusion_learned_v1_gend", name, part, fusion.attack_scores(scores).tolist(),
               v1_checkpoint = checkpoint, weights = fusion.weights, calibration_rows = len(calibration))
        if name == "ffpp":
            report("layernorm_clip_v1", name, part, scores[:, 0].tolist(), checkpoint = checkpoint)
            report("gend_clip_l14", name, part, scores[:, 1].tolist(), face_scale = GEND_FACE_SCALE)